In [ ]:
#1.	Top Revenue Generating Customers - Find top 10 customers contributing highest revenue.

SELECT
    c.customer_id,
    c.customer_name,
    SUM(o.sales_amount) AS total_revenue
FROM customers_view c
JOIN orders_view o
    ON c.customer_id = o.customer_id
GROUP BY
    c.customer_id,
    c.customer_name
ORDER BY total_revenue DESC
LIMIT 10;

In [ ]:
#2.	Monthly Revenue Trend - Find month-wise revenue trend. (use DATE_FORMAT function to choose YYYY-MM) 

SELECT
    DATE_FORMAT(order_date, 'yyyy-MM') AS year_month,
    SUM(sales_amount) AS monthly_revenue
FROM orders_view
GROUP BY DATE_FORMAT(order_date, 'yyyy-MM')
ORDER BY year_month;

select year_month,sum(sales_amount) as rev from (SELECT
    DATE_FORMAT(order_date, 'yyyy-MM') AS year_month, *
FROM orders_view
ORDER BY year_month) group by year_month

In [ ]:
#3.	Running Total Revenue - Finance team wants cumulative revenue.
SELECT
    order_id,
    order_date,             
    sales_amount,
    SUM(sales_amount) OVER (
        ORDER BY order_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_revenue
FROM orders_view
ORDER BY order_date;

In [ ]:
#4.	Previous Order Analysis - Compare customer current purchase with previous purchase.(use LAG )
SELECT
    customer_id,
    order_id,
    order_date,
    sales_amount,
    
    LAG(sales_amount) OVER (
        PARTITION BY customer_id
        ORDER BY order_date
    ) AS previous_purchase,

    sales_amount -
    LAG(sales_amount) OVER (
        PARTITION BY customer_id
        ORDER BY order_date
    ) AS difference

FROM orders_view;

In [ ]:
#Next Purchase Prediction using LEAD - Predict next customer purchase amount.
SELECT
    customer_id,
    order_id,
    order_date,
    sales_amount,

    LEAD(sales_amount) OVER (
        PARTITION BY customer_id
        ORDER BY order_date
    ) AS next_purchase_amount

FROM orders_view;

In [ ]:
#6.	Customer Retention Analysis - Find customers who ordered in consecutive months.
WITH customer_months AS (

    SELECT DISTINCT
        customer_id,
        DATE_TRUNC('month', order_date) AS month
    FROM orders_view

),

previous_months AS (

    SELECT
        customer_id,
        month,

        LAG(month) OVER (
            PARTITION BY customer_id
            ORDER BY month
        ) AS previous_month

    FROM customer_months

)

SELECT
    customer_id,
    month,
    previous_month
FROM previous_months
WHERE month = ADD_MONTHS(previous_month, 1);

In [ ]:
#7.	Highest Selling Product Per Category - Find top-selling product in each category.

WITH product_sales AS (

    SELECT
        p.category,
        p.product_id,
        p.product_name,
        SUM(o.quantity) AS total_quantity

    FROM orders_view o
    JOIN products_view p
        ON o.product_id = p.product_id

    GROUP BY
        p.category,
        p.product_id,
        p.product_name
),

ranked_products AS (

    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY category
            ORDER BY total_quantity DESC
        ) AS rn

    FROM product_sales

)

SELECT *
FROM ranked_products
WHERE rn = 1;

In [ ]:
#8.	Average Shipping Delay - Logistics team wants average delivery delay.( find AVG and DATEDIFF ).
SELECT
    customer_id,
    AVG(
        DATEDIFF(ship_date, order_date)
    ) AS average_shipping_delay
FROM orders_view
GROUP BY customer_id;

In [ ]:
#9.	Customers With No Orders - Marketing team wants inactive customers.

SELECT
    c.customer_id,
    c.customer_name
FROM customers_view c
LEFT JOIN orders_view o
    ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL;

In [ ]:
#10.	Rank Customers Based on Revenue- Basically rank customer who generates high revenue.

WITH customer_revenue AS (

    SELECT
        customer_id,
        SUM(sales_amount) AS total_revenue
    FROM orders_view
    GROUP BY customer_id

)

SELECT
    customer_id,
    total_revenue,

    RANK() OVER (
        ORDER BY total_revenue DESC
    ) AS revenue_rank

FROM customer_revenue;

In [ ]:
#11.	Detect Revenue Drop - Find customers whose sales reduced compared to previous purchase.(use LAG and later filter curr sales < prev sales)
WITH sales_comparison AS (

    SELECT
        customer_id,
        order_id,
        order_date,
        sales_amount,

        LAG(sales_amount) OVER (
            PARTITION BY customer_id
            ORDER BY order_date
        ) AS previous_sales

    FROM orders_view

)

SELECT *
FROM sales_comparison
WHERE sales_amount < previous_sales;

In [ ]:
#Day-wise Revenue Analysis

SELECT
    DATE_FORMAT(order_date, 'EEEE') AS weekday,
    SUM(sales_amount) AS total_revenue
FROM orders_view
GROUP BY DATE_FORMAT(order_date, 'EEEE')
ORDER BY total_revenue DESC;

In [ ]:
#7-Day Moving Average

WITH daily_sales AS (

    SELECT
        order_date,
        SUM(sales_amount) AS daily_revenue
    FROM orders_view
    GROUP BY order_date
)

SELECT
    order_date,
    daily_revenue,

    AVG(daily_revenue) OVER (
        ORDER BY order_date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS moving_avg_7_days

FROM daily_sales
ORDER BY order_date;

In [ ]:
#Revenue Contribution %

WITH customer_revenue AS (

    SELECT
        customer_id,
        SUM(sales_amount) AS customer_revenue
    FROM orders_view
    GROUP BY customer_id

)

SELECT
    customer_id,
    customer_revenue,

    ROUND(
        customer_revenue /
        SUM(customer_revenue) OVER () * 100,
        2
    ) AS contribution_percentage

FROM customer_revenue;

In [ ]:
#Fraudulent Orders
WITH previous_orders AS (

    SELECT
        customer_id,
        order_id,
        order_date,
        sales_amount,

        LAG(order_date) OVER (
            PARTITION BY customer_id
            ORDER BY order_date
        ) AS previous_order_time

    FROM orders_view

),
fraud_check AS (

    SELECT
        *,
        TIMESTAMPDIFF(
            SECOND,
            previous_order_time,
            order_date
        ) AS time_difference

    FROM previous_orders
)

SELECT *
FROM fraud_check
WHERE time_difference <= 300
  AND sales_amount > 50000;------------- take the max purchase value

In [ ]:
#Customer Churn — Inactive for 90 Days

WITH last_purchase AS (

    SELECT
        customer_id,
        MAX(order_date) AS last_order_date
    FROM orders_view
    GROUP BY customer_id

),

latest_date AS (

    SELECT
        MAX(order_date) AS max_order_date
    FROM orders_view

)

SELECT
    c.customer_id,
    c.customer_name,
    lp.last_order_date

FROM customers_view c

LEFT JOIN last_purchase lp
    ON c.customer_id = lp.customer_id

CROSS JOIN latest_date l

WHERE lp.last_order_date IS NULL
   OR DATEDIFF(
        l.max_order_date,
        lp.last_order_date
      ) > 90;

In [ ]:
#Product Affinity Analysis

SELECT
    a.product_id AS product1,
    b.product_id AS product2,
    COUNT(*) AS times_bought_together

FROM orders_view a

JOIN orders_view b
    ON a.order_id = b.order_id
   AND a.product_id < b.product_id

GROUP BY
    a.product_id,
    b.product_id

ORDER BY times_bought_together DESC;

In [ ]:
#Peak Sales Hour

SELECT
    HOUR(order_date) AS order_hour,
    COUNT(order_id) AS order_count

FROM orders_view

GROUP BY HOUR(order_date)

ORDER BY order_count DESC

LIMIT 1;

In [ ]:
#Second Highest Revenue Customer Per State
WITH customer_revenue AS (

    SELECT
        c.state,
        c.customer_id,
        c.customer_name,
        SUM(o.sales_amount) AS total_revenue

    FROM customers_view c

    JOIN orders_view o
        ON c.customer_id = o.customer_id

    GROUP BY
        c.state,
        c.customer_id,
        c.customer_name

),

ranked_customers AS (

    SELECT
        *,
        DENSE_RANK() OVER (
            PARTITION BY state
            ORDER BY total_revenue DESC
        ) AS revenue_rank

    FROM customer_revenue

)

SELECT *
FROM ranked_customers
WHERE revenue_rank = 2;

In [ ]:
#revenue spike

WITH customer_average AS (

    SELECT
        order_id,
        customer_id,
        sales_amount,

        AVG(sales_amount) OVER (
            PARTITION BY customer_id
        ) AS customer_avg_sales

    FROM orders_view

)

SELECT *
FROM customer_average
WHERE sales_amount > customer_avg_sales;